In [5]:
!pip install polars duckdb

   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
    --------------------------------------- 0.3/12.3 MB ? eta -:--:--
    --------------------------------------- 0.3/12.3 MB ? eta -:--:--
    --------------------------------------- 0.3/12.3 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.3 MB 479.2 kB/s eta 0:00:25
   - -------------------------------------- 0.5/12.3 MB 479.2 kB/s eta 0:00:25
   -- ------------------------------------- 0.8/12.3 MB 516.0 kB/s eta 0:00:23
   -- ------------------------------------- 0.8/12.3 MB 516.0 kB/s eta 0:00:23
   --- ------------------------------------ 1.0/12.3 MB 524.3 kB/s eta 0:00:22
   --- ------------------------------------ 1.0/12.3 MB 524.3 kB/s eta 0:00:22
   ---- ----------------------------


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime
import polars as pl
import duckdb
import glob
import duckdb
import gc
import os

pd.set_option('display.max_columns',999)

#### Carregando os dataset

In [7]:
df_atlas = pd.read_csv('raw/mundo_onu_adh_municipio (1).csv')

In [8]:
df_lept = pd.read_parquet('processed/df_hepat_processed_v1.parquet')

#### Select columns Atlas

In [9]:
columns_atlas = ["id_municipio","expectativa_vida","fecundidade_total","prob_sobrevivencia_40","prob_sobrevivencia_60","taxa_freq_bruta_basico","taxa_freq_bruta_fundamental","taxa_freq_bruta_medio","taxa_freq_liquida_superior","prop_pobreza_extrema","prop_vulner_pobreza","taxa_agua_encanada","taxa_banheiro_agua_encanada","taxa_coleta_lixo","taxa_energia_eletrica","taxa_agua_esgoto_inadequados","taxa_paredes_inadequados","populacao","indice_escolaridade","indice_frequencia_escolar","idhm","idhm_e","idhm_l","idhm_r"]

In [10]:
df_atlas=df_atlas[columns_atlas]

### Make df_train

In [12]:
df_train=df_lept.copy()

In [13]:
df_train.columns = [col.lower() for col in df_train.columns ]
df_train['id_municip']=df_train.id_municip.astype(int)
df_train.rename({'id_municip':'id_municipio'},axis=1,inplace=True)

In [14]:
df_train

,tp_not,id_agravo,dt_notific,sem_not,nu_ano,sg_uf_not,id_municipio,id_regiona,id_unidade,dt_sin_pri,...,baixa_escolaridade,raca_vulneravel,faixa_vulneravel,vulnerabilidade_social,nivel_vulnerabilidade,alta_vulnerabilidade,mes_sintomas,semana_sintomas,estacao,periodo_chuvoso
0,Surto,Leptospirose nao especificada,1420070400000000000,201453,2015,Minas Gerais (MG),317020,1462,5617286,2014-12-28,...,0,0,1,1,Moderada,0,12,52,Verão,1
1,Surto,Leptospirose nao especificada,1420070400000000000,201453,2015,Acre (AC),120040,1938,2001578,2014-12-26,...,0,1,1,2,Alta,1,12,52,Verão,1
2,Surto,Leptospirose nao especificada,1420070400000000000,201453,2015,Rio Grande do Sul (RS),431730,1593,2232715,2015-01-01,...,1,0,0,1,Moderada,0,1,1,Verão,1
3,Surto,Leptospirose nao especificada,1420070400000000000,201453,2015,Paraná (PR),412550,1356,2753278,2014-12-26,...,0,0,0,0,Baixa,0,12,52,Verão,1
4,Surto,Leptospirose nao especificada,1420070400000000000,201453,2015,São Paulo (SP),351300,1335,6212573,2014-12-29,...,0,0,0,0,Baixa,0,12,1,Verão,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27515,Surto,Leptospirose,1735603200000000000,202501,2024,Espírito Santo (ES),320120,Não informado,25478,2024-12-19,...,0,0,1,1,Moderada,0,12,51,Verão,1
27516,Surto,Leptospirose,1735603200000000000,202501,2024,Espírito Santo (ES),320010,Não informado,24028,2024-12-23,...,0,0,0,0,Baixa,0,12,52,Verão,1
27517,Surto,Leptospirose,1735603200000000000,202501,2024,Espírito Santo (ES),320010,Não informado,24028,2024-12-27,...,0,0,1,1,Moderada,0,12,52,Verão,1
27518,Surto,Leptospirose,1735257600000000000,202452,2024,Espírito Santo (ES),320520,Não informado,25469,2024-12-24,...,0,1,0,1,Moderada,0,12,52,Verão,1


In [15]:
df_atlas['id_municipio'] = df_atlas['id_municipio']//10
df_atlas_recent=df_atlas.groupby('id_municipio').tail(1)

In [16]:
df_merge=pd.merge(df_train,df_atlas_recent,on=['id_municipio'],how='inner')

In [18]:
df_merge

,tp_not,id_agravo,dt_notific,sem_not,nu_ano,sg_uf_not,id_municipio,id_regiona,id_unidade,dt_sin_pri,sem_pri,ano_nasc,nu_idade_n,cs_sexo,cs_gestant,cs_raca,cs_escol_n,sg_uf,id_mn_resi,id_rg_resi,id_pais,nduplic_n,dt_digita,dt_transus,dt_transdm,dt_transsm,dt_transrs,dt_transse,nu_lote_v,dt_invest,id_ocupa_n,ant_cb_lam,ant_cb_cri,ant_cb_cai,ant_cb_fos,ant_cb_sin,ant_cb_pla,ant_cb_cor,ant_cb_roe,ant_cb_gra,ant_cb_ter,ant_cb_lix,ant_cb_out,ant_ou_des,ant_humano,ant_animai,cli_dt_ate,cli_febre,cli_mialgi,cli_cefale,cli_prost,cli_conges,cli_pantur,cli_vomito,cli_diarre,cli_icteri,cli_renal,cli_respir,cli_cardia,cli_hemopu,cli_hemorr,cli_mening,cli_outros,cli_otrdes,ate_hosp,ate_dt_int,ate_dt_alt,ate_uf,ate_munici,lab_dt_1,lab_elis_1,lab_dt_2,lab_elis_2,dtmicro1,micro1_s1,micro1_t_1,micro1_s_2,micro1_t_2,lab_micr_1,dtmicro2,micro2_s1,micro2_t_1,micro2_s_2,micro2_t_2,lab_micr_2,dtisola,res_isol,dtimuno,res_imuno,dt_pcr,res_pcr,classi_fin,criterio,tpautocto,coufinf,copaisinf,comuninf,con_area,con_ambien,doenca_tra,evolucao,dt_obito,dt_encerra,dt_risco1,dt_risco2,co_mun_r1,co_mun_r2,co_mun_r3,co_mun_r4,co_uf_r1,co_uf_r2,co_uf_r3,co_uf_r4,ano,idade_anos,faixa_etaria,classi_fin_valido,evolucao_valido,baixa_escolaridade,raca_vulneravel,faixa_vulneravel,vulnerabilidade_social,nivel_vulnerabilidade,alta_vulnerabilidade,mes_sintomas,semana_sintomas,estacao,periodo_chuvoso,expectativa_vida,fecundidade_total,prob_sobrevivencia_40,prob_sobrevivencia_60,taxa_freq_bruta_basico,taxa_freq_bruta_fundamental,taxa_freq_bruta_medio,taxa_freq_liquida_superior,prop_pobreza_extrema,prop_vulner_pobreza,taxa_agua_encanada,taxa_banheiro_agua_encanada,taxa_coleta_lixo,taxa_energia_eletrica,taxa_agua_esgoto_inadequados,taxa_paredes_inadequados,populacao,indice_escolaridade,indice_frequencia_escolar,idhm,idhm_e,idhm_l,idhm_r
0,Surto,Leptospirose nao especificada,1420070400000000000,201453,2015,Minas Gerais (MG),317020,1462,5617286,2014-12-28,201453,2003,11.0,Masculino,Não se aplica,Não informado,Não informado,MG,317020,1462,1,Não informado,20150102,Não informado,Não informado,20150108,Não informado,Não informado,0000000,20150101,Não informado,Não informado,Não informado,Não informado,Não informado,Não informado,Não informado,Não informado,Sim,Sim,Não informado,Não informado,Não informado,Não informado,Não informado,Não informado,20150101,Sim,Sim,Sim,Não,Não,Não,Não,Não,Não,Não,Sim,Não,Não,Não,Não,Não,Não informado,Sim,20150101,Não informado,MG,317020,20150102,Não reagente,Não informado,Não Realizado,Não informado,Não informado,Não informado,Não informado,Não informado,Inconclusivo,Não informado,Não informado,Não informado,Não informado,Não informado,Inconclusivo,Não informado,Não Realizado,Não informado,Não Realizado,Não informado,Não Realizado,Descartado,Laboratorial,Não informado,Não informado,0,Não informado,Não informado,Não informado,Não informado,Cura,Não informado,20150120,Não informado,Não informado,Não informado,Não informado,Não informado,Não informado,Não informado,Não informado,Não informado,Não informado,2015,11.0,1-12 anos,NaN,Cura,0,0,1,1,Moderada,0,12,52,Verão,1,78.09,1.7,95.28,86.69,101.77,110.28,79.42,25.24,0.70,12.41,99.52,98.97,99.85,99.92,0.18,0.5,604013,0.646,0.754,0.789,0.716,0.885,0.776
1,Surto,Leptospirose nao especificada,1421020800000000000,201502,2015,Minas Gerais (MG),317020,1462,2152924,2014-12-15,201451,1996,18.0,Masculino,Não se aplica,Branca,Não informado,MG,317020,1462,1,Não informado,20150113,Não informado,Não informado,20150205,Não informado,Não informado,0000000,20150112,Não informado,Não,Sim,Não,Não,Sim,Não,Não,Não,Não,Não,Não,Não,Não informado,Não informado,Não informado,20141215,Sim,Sim,Sim,Não,Não,Não,Sim,Sim,Não,Não,Sim,Não,Sim,Sim,Não,Sim,DOR ABDOMINAL,Sim,20141222,Não informado,MG,317020,20141222,Reagente,Não informado,Não Realizado,20141222,1 - ANDAMANA,100,26 - SHERMANI,100,Reagente,Não informado,Não informado,Não informado,Não informado,Não informado,Inconclusivo,Não informado,Não Realizado,Não informado,Não Realiza

### Merge dados Ocupação Ministério do Trabalho

In [19]:
# Baixe ou clone o repositório datasets-br/cbo e carregue o arquivo
url = "https://raw.githubusercontent.com/datasets-br/cbo/master/data/lista.csv"
cbo = pd.read_csv(url, dtype={"codigo": str})
cbo['codigo'] = cbo['codigo'].apply(lambda x: x.replace('-',''))
#cbo['codigo'] = pd.to_numeric(cbo['codigo'],errors='coerce') 
# Criar dicionário código → termo
mapa_cbo = dict(zip(cbo["codigo"], cbo["termo"]))

def mapear_ocupacao(df, coluna="id_ocupa_n"):
    df = df.copy()
    df[coluna] = df[coluna].map(mapa_cbo)
    df[coluna] =df[coluna].fillna('Nao informado')
    return df


In [20]:
df_merge=mapear_ocupacao(df_merge, coluna="id_ocupa_n")

In [26]:
df_merge

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 169926 entries, 0 to 169925
Columns: 151 entries, tp_not to idhm_r
dtypes: UInt32(1), category(2), datetime64[ns](1), float64(25), int32(8), int64(4), object(110)
memory usage: 187.8+ MB


Save file

In [27]:
df_merge.to_parquet('processed/df_lept_atlas_ocup.parquet')

In [36]:
df_merge.to_csv('processed/df_lept_atlas_ocup.csv',index=False)

In [38]:
cbo.to_csv('raw/lista_cbo.csv')